# [M4] AI NHÌN VÀ HIỂU DỮ LIỆU
## Buổi S07: Trích Xuất Đặc Trưng Văn Bản: Bag-of-Words & TF-IDF với Scikit-Learn

---

### Metadata Khóa Học
* **Module:** M4 — AI NHÌN VÀ HIỂU DỮ LIỆU
* **Buổi:** S07
* **Thời lượng dự kiến:** 120 phút
* **Sản phẩm đầu ra (Session Output):** Notebook trích xuất đặc trưng TF-IDF và phân loại văn bản Baseline
* **Mức độ liên quan VOAI / Olympic AI:** Mô hình Baseline số 1 cho mọi bài toán văn bản trong các cuộc thi AI.

#### Mục tiêu học tập (Learning Objectives)
- Hiểu mô hình Không gian Vectơ (Vector Space Model) và phương pháp Túi từ (Bag-of-Words - BoW).
- Nắm vững công thức toán học và trực giác của chỉ số Tần suất từ - Nghịch đảo tần suất tài liệu (TF-IDF).
- Hiểu cấu trúc Ma trận thưa (Sparse Matrix) và lý do tại sao nó tiết kiệm hàng Gigabytes bộ nhớ RAM.
- Xây dựng hoàn chỉnh một Pipeline phân loại văn bản (Text Classification) kết hợp TF-IDF và Logistic Regression bằng Scikit-Learn.

#### Điều kiện tiên quyết (Prerequisites)
- Hoàn thành Buổi M4_S06 (Tokenization & Vocabulary).
- Đã nắm vững Logistic Regression từ Module M2.

---


## 1. WHY? — TẠI SAO PHẢI HỌC CHỦ ĐỀ NÀY?


Trước khi mạng nơ-ron Transformer hay BERT ra đời, làm thế nào mà các kỹ sư AI có thể phân loại hàng triệu tài liệu pháp lý, lọc hàng tỷ email rác hoặc phân tích cảm xúc đánh giá sản phẩm một cách tức thời?

Câu trả lời chính là **TF-IDF (Term Frequency - Inverse Document Frequency)**. Đây là thuật toán trích xuất đặc trưng văn bản thanh lịch và hiệu quả nhất trong lịch sử Khoa học Máy tính. Trong thi đấu Olympic AI, TF-IDF kết hợp với các mô hình tuyến tính (Logistic Regression / Linear SVM) luôn là **Mô hình Baseline số 1 bắt buộc phải dựng đầu tiên** cho mọi bài toán NLP: Nó chạy siêu nhanh (chưa tới 3 giây) nhưng mang lại độ chính xác đáng kinh ngạc, đủ sức đánh bại nhiều mô hình Deep Learning nếu dữ liệu ít!



## 2. CONCEPT — NỀN TẢNG LÝ THUYẾT & TRỰC GIÁC


### 1. Túi từ (Bag-of-Words - BoW) & Hạn chế
* Biểu diễn mỗi văn bản thành một vector có độ dài bằng toàn bộ kích thước từ điển ($V$). Mỗi phần tử đếm số lần từ đó xuất hiện trong văn bản.
* *Nhược điểm lớn:* Các từ cực kỳ phổ biến như "và", "là", "thì" xuất hiện liên tục sẽ chiếm trọng số khổng lồ nhưng lại không mang thông tin phân biệt thể loại!

### 2. Công thức toán học của TF-IDF
TF-IDF giải quyết triệt để nhược điểm của BoW bằng cách nhân hai thành phần:
$$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D)$$
1. **TF (Term Frequency - Tần suất từ trong tài liệu $d$):** Từ $t$ xuất hiện càng nhiều trong văn bản $d$ thì giá trị càng cao.
2. **IDF (Inverse Document Frequency - Nghịch đảo tần suất xuất hiện trong toàn bộ kho tài liệu $D$):**
   $$\text{IDF}(t, D) = \log\left( \frac{1 + |D|}{1 + \text{DF}(t)} \right) + 1$$
   - Nếu từ $t$ xuất hiện trong hầu hết mọi văn bản (ví dụ từ "và"), $\text{DF}(t) \approx |D| \implies \text{IDF} \to \text{rất nhỏ}$ (bị phạt nặng).
   - Nếu từ $t$ chỉ xuất hiện trong đúng 1-2 văn bản chuyên biệt (ví dụ từ "viêm phổi"), $\text{DF}(t)$ nhỏ $\implies \text{IDF} \to \text{rất lớn}$!
* *Ý nghĩa triết học:* **Một từ quan trọng là từ xuất hiện nhiều trong tài liệu hiện tại, nhưng hiếm khi xuất hiện ở các tài liệu khác!**

### 3. Ma trận Thưa (Sparse Matrix)
Do mỗi câu chỉ chứa vài chục từ trong khi từ điển có thể chứa 50,000 từ, hơn 99% các phần tử trong ma trận đều là số 0! Thư viện Scikit-Learn lưu trữ dưới dạng ma trận thưa (CSR Matrix - Compressed Sparse Row), chỉ lưu tọa độ các ô khác 0 để tiết kiệm bộ nhớ.



## 3. VISUAL INTUITION — TRỰC QUAN HÓA KIẾN THỨC


```
Trực quan hóa Cơ chế Trọng số TF-IDF:
   Từ vựng        Tần suất trong câu (TF)   Xuất hiện ở các tài liệu khác (DF)   Trọng số TF-IDF
  "tôi"                 Cao (5 lần)               Rất phổ biến (99% văn bản)       RẤT THẤP ⬇️ (Phạt)
  "thích"               Vừa (2 lần)               Phổ biến (60% văn bản)           TRUNG BÌNH
  "trường_học"          Cao (4 lần)               Rất hiếm (2% văn bản)            RẤT CAO ⬆️ (Từ khóa vàng!)
```



## 4. CODE IMPLEMENTATION & 5. WALKTHROUGH

*Toàn bộ code dưới đây được thiết kế thực tế, có thể chạy độc lập không phụ thuộc tải dữ liệu bên ngoài.*

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# 1. Tao tap du lieu mau ve Phan tich cam xuc danh gia phim (Sentiment Analysis)
corpus_data = [
    ("Phim nay xem cuc ky hay va xuc dong tuyet voi", 1),       # Tich cuc (1)
    ("Dien vien dien xuat qua dinh, kich ban xuat sac", 1),      # Tich cuc (1)
    ("Phim chan ngat, phi tien ve xem phim nay", 0),             # Tieu cuc (0)
    ("Kich ban roi ram vo ly, dien vien do te hai", 0),          # Tieu cuc (0)
    ("Hinh anh am thanh rat dep mat, rat dang tien", 1),         # Tich cuc (1)
    ("That vong toan tap, khong dang xem mot chut nao", 0),      # Tieu cuc (0)
    ("Mot tac pham nghe thuat hoan hao va y nghia", 1),         # Tich cuc (1)
    ("Qua buon ngu, noi dung nhat nheo va te nhat", 0)           # Tieu cuc (0)
]

df_nlp = pd.DataFrame(corpus_data, columns=['Text', 'Sentiment'])
print("===== DU LIEU VAN BAN MAU (8 BAN GHI) =====")
print(df_nlp)

### 🔍 Code Walkthrough (Khối 1)
Khối code chuẩn bị bài toán phân tích cảm xúc (Sentiment Analysis) kinh điển của NLP:
- Tập ngữ liệu gồm các câu đánh giá tích cực (nhãn 1) và tiêu cực (nhãn 0).
- Chứa các từ mang sắc thái cảm xúc rõ ràng làm cơ sở kiểm thử thuật toán.


In [ ]:
# 2. Trich xuat dac trung bang TfidfVectorizer va Huan luyen Mo hinh
# Khoi tao bo trich xuat TF-IDF (ho tro ca tu don Unigram va cap tu Bigram)
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=50)

# Bien doi van ban thanh Ma tran dac trung thuc so
X_tfidf = tfidf.fit_transform(df_nlp['Text'])
y = df_nlp['Sentiment']

print("===== THONG TIN MA TRAN THUA (SPARSE MATRIX) =====")
print(f"Kich thuoc ma tran TF-IDF: {X_tfidf.shape} (8 cau, {X_tfidf.shape[1]} dac trung tu vung)")
print(f"Kieu du lieu luu tru: {type(X_tfidf)} (Chi luu cac gia tri khac 0)")

# 3. Huan luyen mo hinh Logistic Regression tren dac trung TF-IDF
clf_text = LogisticRegression()
clf_text.fit(X_tfidf, y)

# 4. Kiem thu du doan tren cau moi hoan toan
new_reviews = [
    "Dien vien dien xuat rat tuyet voi va hay", # Tich cuc
    "Phim xem chan ngat va that vong qua"       # Tieu cuc
]
new_features = tfidf.transform(new_reviews) # Chi transform, KHONG fit lai!
predictions = clf_text.predict(new_features)

print("\n===== KET QUA DU DOAN TREN VAN BAN MOI =====")
for rev, pred in zip(new_reviews, predictions):
    label = "Tich Cuc (1)" if pred == 1 else "Tieu Cuc (0)"
    print(f"Review: '{rev}' -> Du doan: {label}")

### 🔍 Code Walkthrough (Khối 2)
Khối code xây dựng hoàn chỉnh một NLP Pipeline bằng Scikit-Learn:
- `TfidfVectorizer(ngram_range=(1, 2))`: Tự động tách cả từ đơn và cặp từ liền kề.
- Trích xuất ma trận đặc trưng thưa siêu tốc độ.
- Huấn luyện `LogisticRegression` và chạy suy diễn chính xác trên các câu đánh giá mới.


## 6. PRACTICE — BÀI TẬP THỰC HÀNH


### Bài 1 (Easy): Khám phá từ vựng học được
In ra danh sách 10 từ vựng đầu tiên cùng chỉ số của chúng bằng lệnh `tfidf.vocabulary_`. Nhận xét xem có các cặp từ ghép (Bigram) nào được tạo ra không.

### Bài 2 (Easy): Tìm các từ khóa có trọng số TF-IDF cao nhất
Với câu văn bản đầu tiên (`df_nlp['Text'][0]`), hãy in ra danh sách các từ kèm theo giá trị TF-IDF tương ứng sắp xếp giảm dần để xem từ nào được coi là quan trọng nhất trong câu đó.

### Bài 3 (Medium): So sánh với CountVectorizer (Bag-of-Words thuần)
Import `CountVectorizer` từ `sklearn.feature_extraction.text`. Huấn luyện lại mô hình bằng CountVectorizer thay vì TF-IDF. So sánh điểm số và giải thích tại sao TF-IDF thường cho kết quả phân loại ổn định hơn BoW.

### Bài 4 (Medium): Thiết lập ngưỡng `min_df` và `max_df`
Tìm hiểu 2 siêu tham số cực kỳ quan trọng trong TfidfVectorizer:
- `min_df=2`: Loại bỏ các từ quá hiếm (chỉ xuất hiện trong < 2 văn bản).
- `max_df=0.8`: Loại bỏ các từ quá phổ biến (xuất hiện trong > 80% văn bản).
Áp dụng lên một tập dữ liệu lớn và quan sát số lượng cột đặc trưng giảm đi như thế nào.

### Bài 5 (Challenge): Trích xuất đặc trưng phân biệt nhất của từng lớp (Top Discriminating Words)
Truy cập vào thuộc tính `clf_text.coef_[0]` của mô hình Logistic Regression. Tìm 3 từ vựng có hệ số dương lớn nhất (đặc trưng mạnh nhất cho nhãn Tích cực) và 3 từ có hệ số âm lớn nhất (đặc trưng mạnh nhất cho nhãn Tiêu cực).



## 7. VOAI CONNECTION — CHIẾN LƯỢC PHÒNG THI OLYMPIC


Trong 15 phút đầu tiên khi nhận một đề thi NLP tại Olympic AI, **hãy luôn luôn chạy một Baseline bằng TF-IDF + Logistic Regression/LightGBM**. Một lỗi chết người của nhiều thí sinh là vội vàng bật máy nạp các mô hình ngôn ngữ lớn (LLMs/Transformers) mất 40 phút chỉ để tải trọng số, để rồi sau đó phát hiện máy bị tràn RAM và không kịp có bất kỳ một kết quả nào nộp lên bảng xếp hạng. Một baseline TF-IDF chuẩn chỉ sẽ đảm bảo bạn có điểm số an toàn ngay từ vạch xuất phát!



## 8. DEBUGGING CHALLENGE — BẮT LỖI THỰC CHIẾN


**Tình huống thực tế:** Thí sinh gọi lại phương thức `fit_transform()` trên tập kiểm tra (Test Set):

```python
# CODE CÓ LỖI LỆCH TỪ ĐIỂN VÀ RÒ RỈ DỮ LIỆU:
# X_train_tfidf = tfidf.fit_transform(train_texts)
# X_test_tfidf = tfidf.fit_transform(test_texts) # SAI LẦM CHẾT NGƯỜI!
# Goi fit_transform tren tap Test se tao ra mot tu dien hoan toan moi,
# khien thu tu cot dac trung cua tap Test bi lech 100% so voi tap Train!
```

**Nhiệm vụ của bạn:**
1. Giải thích tại sao bắt buộc phải dùng `tfidf.transform(test_texts)` trên tập Test.
2. Nêu hậu quả nghiêm trọng nếu ma trận đặc trưng của tập Test bị lệch chỉ số cột so với tập Train khi đưa vào mô hình dự đoán.



## 9. KNOWLEDGE CHECK — KIỂM TRA ĐỘ HIỂU BẢN CHẤT


1. Thuật ngữ IDF (Inverse Document Frequency) trong TF-IDF đo lường điều gì?
   - A. Tốc độ gõ phím của người viết văn bản.
   - B. Mức độ hiếm có và giá trị thông tin của một từ vựng trong toàn bộ kho tài liệu: Từ nào càng xuất hiện ở nhiều văn bản khác nhau thì giá trị IDF càng bị kéo giảm về 0.
   - C. Số lượng ký tự trong một từ.
   - D. Tần số quét của màn hình.

2. Tại sao thư viện Scikit-Learn lại lưu trữ ma trận TF-IDF dưới dạng Ma trận thưa (Sparse Matrix)?
   - A. Vì thư viện bị thiếu bộ nhớ.
   - B. Vì trong bài toán văn bản, đại đa số các từ vựng trong từ điển đều không xuất hiện trong một câu cụ thể (hơn 99% phần tử là số 0), việc chỉ lưu các ô khác 0 giúp tiết kiệm hàng Gigabytes RAM.
   - C. Để làm cho code chạy khó hiểu hơn.
   - D. Do định dạng của file CSV.

3. Tham số `ngram_range=(1, 2)` trong `TfidfVectorizer` có ý nghĩa gì?
   - A. Chỉ lấy các câu có độ dài từ 1 đến 2 từ.
   - B. Trích xuất cả các từ đơn lẻ (Unigrams) và các cặp hai từ đi liền kề nhau (Bigrams) làm đặc trưng.
   - C. Lặp lại việc huấn luyện 2 lần.
   - D. Chia tập dữ liệu thành 2 nửa.

4. Giữa từ "và" (xuất hiện ở 100% bài báo) và từ "lượng tử" (chỉ xuất hiện ở 1% bài báo khoa học), từ nào sẽ có chỉ số IDF cao hơn vượt trội?
   - A. Từ "và".
   - B. Cả hai từ có IDF bằng nhau.
   - C. Từ "lượng tử".
   - D. Không xác định được.

5. Trong các cuộc thi AI, mô hình Baseline được khuyến nghị hàng đầu cho bài toán phân loại văn bản dạng bảng là gì?
   - A. Huấn luyện một mạng nơ-ron Transformer 100 lớp từ đầu.
   - B. TfidfVectorizer kết hợp Logistic Regression hoặc Linear SVM.
   - C. Hệ chuyên gia đếm từ bằng tay.
   - D. Dự đoán ngẫu nhiên.



## 10. SESSION SUMMARY — TỔNG KẾT BUỔI HỌC


* **What I learned:**
  - Mô hình Túi từ (Bag-of-Words) và hạn chế của việc đếm từ đơn thuần.
  - Bản chất toán học của TF-IDF và ý nghĩa triết học về tầm quan trọng của từ khóa.
  - Cấu trúc tối ưu bộ nhớ của Ma trận thưa (Sparse Matrix).
* **What I can now do:**
  - Xây dựng một Text Classification Pipeline hoàn chỉnh bằng Scikit-Learn trong chưa đầy 10 dòng code.
  - Nhanh chóng dựng Baseline chất lượng cao cho bất kỳ bài thi NLP nào.
* **What I still need to practice:**
  - Tiến sâu vào biểu diễn ngữ nghĩa liên tục bằng **Word Embeddings & `nn.Embedding`** ở Buổi 8.



## 11. NEXT SESSION — BƯỚC ĐI TIẾP THEO


Trong **Buổi M4_S08**, chúng ta sẽ bước vào không gian ngữ nghĩa học sâu: **Trực Giác Không Gian Vectơ Từ (Word Embeddings) & `nn.Embedding` Trong PyTorch** — Khám phá cách máy tính hiểu được "Vua - Đàn ông + Phụ nữ = Nữ hoàng" thông qua các vector dày đặc (Dense Vectors)!



---

## 🎯 Syllabus Traceability (Đối chiếu Syllabus)

Buổi học này đóng góp trực tiếp vào mục tiêu của **M4**:
- Túi từ (Bag-of-Words) & Bản chất toán học TF-IDF ✓
- Ma trận thưa (Sparse Matrix) & TfidfVectorizer ✓
- Text Classification Pipeline hoàn chỉnh với Scikit-Learn ✓

* **Session Deliverable:** Mã nguồn phân loại văn bản Baseline bằng TF-IDF và Logistic Regression
* **Đảm bảo chuẩn Syllabus Lock:** Nội dung nằm đúng chuẩn 100%, không lan man, giải quyết trực tiếp năng lực cốt lõi.
